In [ ]:
%%writefile cryptovault.py
import argparse
import argparse
import hashlib
import os
import sys

ENGLISH_FREQS = {
    'A': 0.0817, 'B': 0.0149, 'C': 0.0278, 'D': 0.0425, 'E': 0.1270, 'F': 0.0223,
    'G': 0.0202, 'H': 0.0609, 'I': 0.0697, 'J': 0.0015, 'K': 0.0077, 'L': 0.0403,
    'M': 0.0241, 'N': 0.0675, 'O': 0.0751, 'P': 0.0193, 'Q': 0.0010, 'R': 0.0599,
    'S': 0.0633, 'T': 0.0906, 'U': 0.0276, 'V': 0.0098, 'W': 0.0236, 'X': 0.0015,
    'Y': 0.0197, 'Z': 0.0007
}

def caesar_transform(text: str, shift: int) -> str:
    """Encrypts or decrypts text by shifting characters while preserving case."""
    result = []
    for char in text:
        if char.isalpha():
            start = ord('A') if char.isupper() else ord('a')
            shifted_char = chr(start + (ord(char) - start + shift) % 26)
            result.append(shifted_char)
        else:
            result.append(char)
    return "".join(result)

def calculate_score(text: str) -> float:
    """Scores a text based on how closely it matches English letter frequencies."""
    letters = [char.upper() for char in text if char.isalpha()]
    if not letters:
        return float('inf')
    counts = {ch: 0 for ch in ENGLISH_FREQS}
    for ch in letters:
        counts[ch] += 1
    total_letters = len(letters)
    chi_squared = 0.0
    for ch, expected_prop in ENGLISH_FREQS.items():
        expected_count = expected_prop * total_letters
        observed_count = counts[ch]
        chi_squared += ((observed_count - expected_count) ** 2) / expected_count
    return chi_squared

def crack_ciphertext(ciphertext: str):
    """Tests all 26 shifts and prints the top 3 plaintexts matching English profile."""
    # Strip metadata header if present to avoid messing up the statistical profile
    if ciphertext.startswith("HASH-GUARD::"):
        try:
            _, _, ciphertext = ciphertext.split("\n", 2)
        except ValueError:
            pass

    candidates = []
    for shift in range(26):
        decrypted_candidate = caesar_transform(ciphertext, -shift)
        score = calculate_score(decrypted_candidate)
        candidates.append((score, shift, decrypted_candidate))
    candidates.sort(key=lambda x: x[0])
    
    print("\n--- TOP 3 LIKELY PLAINTEXTS ---")
    for i in range(min(3, len(candidates))):
        score, shift, text = candidates[i]
        print(f"Rank {i+1} (Key/Shift used to decrypt: {shift}):")
        print(f"Text: \"{text}\"\n")

def main():
    parser = argparse.ArgumentParser(description="CryptoVault CLI Tool with Hash Guard.")
    parser.add_argument('mode', choices=['encrypt', 'decrypt', 'crack'], help="Operation mode")
    parser.add_argument('file_or_text', help="Path to text file or raw string")
    parser.add_argument('--shift', type=int, default=0, help="Shift key value")
    parser.add_argument('--verify', action='store_true', help="Enable integrity checks using SHA-256 metadata")

    args = parser.parse_args()

    # Handle Cracking Mode
    if args.mode == 'crack':
        if os.path.exists(args.file_or_text):
            with open(args.file_or_text, 'r', encoding='utf-8') as f:
                content = f.read()
        else:
            content = args.file_or_text
        crack_ciphertext(content)
        return

    # Handle File Input Validation
    if not os.path.exists(args.file_or_text):
        print(f"Error: File '{args.file_or_text}' not found.", file=sys.stderr)
        sys.exit(1)

    with open(args.file_or_text, 'r', encoding='utf-8') as f:
        file_data = f.read()

    # 1. ENCRYPTION MODE
    if args.mode == 'encrypt':
        encrypted_text = caesar_transform(file_data, args.shift)
        
        if args.verify:
            # Compute SHA-256 of original raw file text
            original_hash = hashlib.sha256(file_data.encode('utf-8')).hexdigest()
            # Embed metadata as a header string above payload
            final_output = f"HASH-GUARD::SHA256::{original_hash}\n{encrypted_text}"
            print("[✓] Integrity verification metadata generated and stored.")
        else:
            final_output = encrypted_text
            
        print("\n--- Encrypted Output ---")
        print(final_output)

    # 2. DECRYPTION MODE
    elif args.mode == 'decrypt':
        # Check if text contains our metadata format
        has_metadata = file_data.startswith("HASH-GUARD::")
        
        if args.verify:
            if not has_metadata:
                print("[-] Error: --verify flag used but input file missing Hash Guard metadata.", file=sys.stderr)
                sys.exit(1)
            
            try:
                # Split metadata line from actual cipher text
                header_line, encrypted_payload = file_data.split("\n", 1)
                stored_hash = header_line.split("::")[2]
            except (ValueError, IndexError):
                print("[!] CRITICAL ERROR: Metadata header format is corrupted.", file=sys.stderr)
                sys.exit(1)
                
            # Perform regular decryption
            decrypted_text = caesar_transform(encrypted_payload, -args.shift)
            # Compute fresh hash from newly decrypted data
            computed_hash = hashlib.sha256(decrypted_text.encode('utf-8')).hexdigest()
            
            print("--- Verification Check ---")
            print(f"Stored Hash:   {stored_hash}")
            print(f"Computed Hash: {computed_hash}")
            
            # Direct verification check
            if computed_hash == stored_hash:
                print("[✓] INTEGRITY SUCCESS: Hashes match perfectly. Data is authentic.\n")
            else:
                print("[💥] INTEGRITY WARNING: TAMPERING DETECTED! Hashes do not match. The file was modified.\n")
                
            print("--- Decrypted Output ---")
            print(decrypted_text)
            
        else:
            # Fallback path if file contains metadata but user skips verification parameter
            if has_metadata:
                _, encrypted_payload = file_data.split("\n", 1)
                print(caesar_transform(encrypted_payload, -args.shift))
            else:
                print(caesar_transform(file_data, -args.shift))

if __name__ == "__main__":
    main()
